# 무역 트렌드의 기술적 분석 — 정본 계산기

`연구계획서.md`의 첫 갈래. 성질별 수출입실적(`fact_temper`, 1995.01~)으로 품목군 열다섯의 월 계열을 만들고
트렌드·구조변화·구성·단가물량을 잰다. 마지막 절이 본문 인용 수치를 `chk()`로 대조한다.

| 절 | 단계 | 내용 |
|---|---|---|
| §0 | — | 설정. DB 연결, 대응표 |
| §1 | 1 | 계열 구축. 품목군 열다섯 + 총수출·반도체 제외 + 총수입·에너지 제외·대분류 |
| §2 | 1 | 달력 설명변수. 조업일수, 설·추석 창(20,10), 예측 구간 12개월 |
| §3 | 1 | X-13ARIMA-SEATS(사양 A) 달력·계절조정, 진단, STL 강건성, 사양 B 비교 |
| §4 | 2 | 트렌드 분해. HP(본방법)·UCM(강건성), STL 트렌드는 참고(논문에 없음) |
| §5 | 2 | Bai-Perron 다중 단절(동적 계획법·LWZ), sup-F |
| §6 | 2 | 그림 1·3 |
| §7 | 3 | 구성: 대분류·품목군 비중, 집중도, 국면별 기여도 |
| §8 | 3 | 단가·물량(Törnqvist) |
| §9 | 3 | 변동성·동조성, 그림 2 |
| §10 | 4 | 본문 인용 수치 검증 `chk()` |

실행: `jupyter nbconvert --to notebook --execute --inplace 무역트렌드_기술분석.ipynb --ExecutePreprocessor.kernel_name=kcsdb`

## §0. 설정

In [1]:
import os, sys, re, datetime as dt, warnings
import numpy as np, pandas as pd, duckdb
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from korean_lunar_calendar import KoreanLunarCalendar
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT: raise FileNotFoundError("KCSDB2 루트를 찾지 못했습니다")
    ROOT = up
DB   = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")
CAL  = os.path.join(ROOT, "data", "external", "KASI_공휴일.csv")
HERE = os.path.join(ROOT, "analysis", "무역 트렌드 분석")
CACHE = os.path.join(HERE, "cache"); OUT = os.path.join(HERE, "outputs"); IMG = os.path.join(HERE, "img")
for p in (CACHE, OUT, IMG): os.makedirs(p, exist_ok=True)
MAP = pd.read_csv(os.path.join(OUT, "품목군_대응표_수출.csv"), dtype=str)

YM_START, YM_END = 199501, None      # 끝은 DB가 정한다
con = duckdb.connect(DB, read_only=True)
YM_END = con.execute("SELECT MAX(yyyymm) FROM fact_temper").fetchone()[0]
print("자료 끝:", YM_END, "| 대응표", len(MAP), "부호 →", MAP.group15.nunique(), "품목군")

자료 끝: 202607 | 대응표 153 부호 → 16 품목군


## §1. 계열 구축

품목군 열다섯은 대응표 `group15`로, 총수출과 반도체 제외는 그 합으로 만든다. 수입은 총수입, 에너지 제외(원유·가스·석탄·석유제품을
뺀 것), 대분류 셋(소비재·원자재·자본재)이다. 1996년 수입은 원천 결함(계획서 II.2절)이라 수입 계열은 1997.01부터 쓴다.
금액은 달러, 중량은 kg, 단가는 달러/kg. 캐시 `cache/series_monthly.csv`(긴 형태).

In [2]:
con.register("_map", MAP[["temper_cd", "group15"]])
exp = con.execute("""
    SELECT f.yyyymm, m.group15 AS series, SUM(f.dlr) AS dlr, SUM(f.wgt) AS wgt
    FROM fact_temper f JOIN _map m USING (temper_cd)
    WHERE f.imexp='수출' GROUP BY 1,2""").df()
tot = exp.groupby("yyyymm")[["dlr","wgt"]].sum().reset_index().assign(series="총수출")
ex_semi = exp[exp.series != "반도체"].groupby("yyyymm")[["dlr","wgt"]].sum().reset_index().assign(series="반도체 제외")
imp = con.execute("""
    SELECT f.yyyymm,
           CASE WHEN d.major10 IN ('원유','가스','석탄','석유제품') THEN '에너지' ELSE '비에너지' END AS eng,
           SUBSTR(d.x1, 4) AS x1, SUM(f.dlr) AS dlr, SUM(f.wgt) AS wgt
    FROM fact_temper f JOIN dim_temper d USING (imexp, temper_cd)
    WHERE f.imexp='수입' GROUP BY 1,2,3""").df()
imp_tot = imp.groupby("yyyymm")[["dlr","wgt"]].sum().reset_index().assign(series="총수입")
imp_ex  = imp[imp.eng=="비에너지"].groupby("yyyymm")[["dlr","wgt"]].sum().reset_index().assign(series="에너지 제외 수입")
imp_x1  = imp.groupby(["yyyymm","x1"])[["dlr","wgt"]].sum().reset_index().rename(columns={"x1":"series"})
imp_x1["series"] = "수입 " + imp_x1.series.str.strip()
S = pd.concat([exp, tot, ex_semi, imp_tot, imp_ex, imp_x1], ignore_index=True)
S = S[S.yyyymm.between(YM_START, YM_END)]
S.loc[S.series.str.startswith(("총수입","에너지 제외","수입 ")) & (S.yyyymm < 199701), ["dlr","wgt"]] = np.nan   # 1996 결함
S["uv"] = S.dlr / S.wgt
S = S.sort_values(["series","yyyymm"]).reset_index(drop=True)
S.to_csv(os.path.join(CACHE, "series_monthly.csv"), index=False, encoding="utf-8-sig")

EXP15 = [s for s in MAP.group15.unique() if s != "나머지"]
ORDER = ["총수출","반도체 제외"] + sorted(EXP15, key=lambda s: -S[(S.series==s)&(S.yyyymm>=202301)].dlr.sum()) + ["나머지",
         "총수입","에너지 제외 수입","수입 소비재","수입 원자재","수입 자본재"]
assert set(ORDER) == set(S.series.unique()), set(S.series.unique()) ^ set(ORDER)
# 검증: 열다섯 + 나머지 = 총수출, 계열마다 0인 달이 없다
W = S.pivot(index="yyyymm", columns="series", values="dlr")
assert np.allclose(W[EXP15 + ["나머지"]].sum(axis=1), W["총수출"])
assert (W.drop(columns=["총수입","에너지 제외 수입","수입 소비재","수입 원자재","수입 자본재"]) > 0).all().all()
summ = (S[S.yyyymm.between(202301, 202512)].groupby("series").dlr.sum() / 1e8).round(0).rename("2023~25 억달러")
summ = summ.reindex(ORDER); summ.loc[EXP15 + ["나머지"]] = summ.loc[EXP15 + ["나머지"]]
print(pd.concat([summ, S.groupby("series").yyyymm.agg(["min","max"]).reindex(ORDER)], axis=1).to_string())

           2023~25 억달러     min     max
series                                
총수출            20252.0  199501  202607
반도체 제외         16054.0  199501  202607
반도체             4197.0  199501  202607
화공품             2237.0  199501  202607
승용차             2051.0  199501  202607
석유제품            1491.0  199501  202607
일반기계            1533.0  199501  202607
철강제품            1440.0  199501  202607
선박               757.0  199501  202607
자동차부품            642.0  199501  202607
컴퓨터주변기기          390.0  199501  202607
무선통신기기           538.0  199501  202607
정밀기기             336.0  199501  202607
이차전지             310.0  199501  202607
제조장비             281.0  199501  202607
의약품              248.0  199501  202607
가전제품             234.0  199501  202607
나머지             3567.0  199501  202607
총수입            19062.0  199501  202607
에너지 제외 수입      14341.0  199501  202607
수입 소비재          3128.0  199501  202607
수입 원자재          9224.0  199501  202607
수입 자본재          6710.0  199501  202607


## §2. 달력 설명변수

조업일수는 `dim_workday10d`의 세 구간을 달로 합친 것(월~금, 공휴일 제외). 설·추석 창은 달력효과 연구의 설계를 그대로 옮겼다 —
연휴 사흘(`dur`)과 그 앞 20일(`pre`)·뒤 10일(`post`)이 각 달에 걸치는 비율. 설명변수는 표본(1995.01~2026.07)의 달력월 평균으로 중심화하며,
X-13이 X-11 비대칭 필터에 쓸 12개월 예측 구간(~2027.07)까지 같은 평균으로 중심화해 넘긴다.

In [3]:
wd = con.execute("SELECT base_ym AS ym, SUM(workdays) AS workdays FROM dim_workday10d GROUP BY 1 ORDER BY 1").df().set_index("ym")
hol = set(pd.to_datetime(pd.read_csv(CAL, dtype=str, encoding="utf-8-sig")["date"]).dt.date)

def lunar(y, m, d):
    k = KoreanLunarCalendar(); k.setLunarDate(y, m, d, False)
    return dt.date.fromisoformat(k.SolarIsoFormat())
YEARS = range(1994, 2028)
seol = {y: lunar(y, 1, 1) for y in YEARS}; chuseok = {y: lunar(y, 8, 15) for y in YEARS}

def holiday_regressors(anchor, w_pre, w_post, name):
    rows = []
    for y, d0 in anchor.items():
        dur = [d0 + dt.timedelta(k) for k in (-1, 0, 1)]
        pre = [dur[0] - dt.timedelta(k) for k in range(1, w_pre + 1)]
        post = [dur[-1] + dt.timedelta(k) for k in range(1, w_post + 1)]
        for k, win in [("pre", pre), ("dur", dur), ("post", post)]:
            for d in win: rows.append({"ym": d.year*100 + d.month, "var": f"{name}_{k}", "w": 1/len(win)})
    return pd.DataFrame(rows).groupby(["ym","var"])["w"].sum().unstack(fill_value=0.0)

W1, W2 = 20, 10
LEAD = 12                                    # X-13 예측 구간
YM_LEAD = (YM_END // 100 + (YM_END % 100 + LEAD - 1) // 12) * 100 + (YM_END % 100 + LEAD - 1) % 12 + 1
H = holiday_regressors(seol, W1, W2, "seol").add(holiday_regressors(chuseok, W1, W2, "chu"), fill_value=0.0)
X = wd.join(H).fillna(0.0)
X["ln_wd"] = np.log(X["workdays"])
X = X.loc[(X.index >= YM_START) & (X.index <= YM_LEAD)].copy()
X["month"] = X.index % 100; X["year"] = X.index // 100
CAL_VARS = ["ln_wd","seol_pre","seol_dur","seol_post","chu_pre","chu_dur","chu_post"]
for c in CAL_VARS:
    mm = X.loc[:YM_END].groupby("month")[c].mean(); X[c] = X[c] - X["month"].map(mm)
Xcal = X[CAL_VARS + ["month","year"]]
assert len(Xcal.loc[:YM_END]) == len(W), (len(Xcal.loc[:YM_END]), len(W))
print("설명변수", Xcal.index.min(), "~", Xcal.index.max(), "| 예측 구간 끝", YM_LEAD)
print(Xcal.loc[:YM_END, CAL_VARS].describe().T[["mean","std","min","max"]].round(3))

설명변수 199501 ~ 202707 | 예측 구간 끝 202707


           mean    std    min    max
ln_wd      -0.0  0.054 -0.242  0.115
seol_pre   -0.0  0.124 -0.581  0.581
seol_dur   -0.0  0.187 -0.625  0.625
seol_post   0.0  0.104 -0.650  0.663
chu_pre     0.0  0.095 -0.494  0.516
chu_dur     0.0  0.155 -0.785  0.785
chu_post    0.0  0.178 -0.555  0.555


## §3. 달력조정과 계절조정: X-13ARIMA-SEATS

한국은행·통계청이 쓰는 X-13ARIMA-SEATS(센서스국 v1.1 b62, `C:\tools\x13as`)를 본방법으로 쓴다. 스펙 파일을 직접 써서 실행 파일을 부른다.
사양 A: 로그 변환, 사용자 정의 설명변수 `ln_wd`(usertype=td)와 설·추석 3창(usertype=holiday), ARIMA 자동 선택(automdl), 이상치 자동 탐지(AO·LS),
12개월 예측, X-11 필터(계절 이동평균은 MSR 자동 선택). 자동 선택이 특이 공분산으로 실패하면 항공기 모형 (0 1 1)(0 1 1)로 되돌린다.

출력에서 회귀 계수(조업일수 탄력성 $\beta_D$와 명절 창), 이상치, ARIMA 차수, AICc, D8 계절성 $F$검정, M7, Q, QS 통계량을 읽고
저장 표 d10(계절 요인)·d11(계절조정)·d12(트렌드-순환)·d13(불규칙)·td·hol(달력 요인)을 읽는다. 계절조정 계열은 $\ln$ d11, 달력조정 계열은 $\ln y - \ln(\mathrm{td}) - \ln(\mathrm{hol})$.
**M7 ≥ 1이면 계절조정을 쓰지 않고 달력조정 계열만 쓴다**(X-11 품질 기준: 이동 계절성이 안정 계절성을 압도). 강건성으로 같은 달력조정 계열에 STL(주기 12, robust, 계절창 13)을 얹어 견준다.

사양 B(센서스 표준: 요일 6계수 + 평일 공휴일 수 + 명절 창)와의 비교는 총수출에서만 한다. 본문 II.3절이 그 차이를 적는다.

In [4]:
import subprocess
X13 = r"C:\tools\x13as\x13as.exe"
XWORK = os.path.join(CACHE, "x13"); os.makedirs(XWORK, exist_ok=True)
assert os.path.exists(X13), "X-13 실행 파일이 없다: " + X13

def x13_run(tag, y, exog, log=True, maxlead=LEAD, arima=None):
    """y: Series(index=yyyymm). exog: DataFrame(index=yyyymm, 예측 구간까지). 결과 dict."""
    base = os.path.join(XWORK, tag); sy, sm = divmod(int(y.index.min()), 100)
    with open(base + ".dat", "w") as f:
        for v in y.values: f.write(f"{v:.6f}\n")
    with open(base + "_x.dat", "w") as f:
        for row in exog.values: f.write(" ".join(f"{v:.8f}" for v in row) + "\n")
    types = " ".join(["td"] + ["holiday"] * (exog.shape[1] - 1))
    spc = "\n".join([
        f'series{{ title="{tag}" start={sy}.{sm:02d} period=12 file="{tag}.dat" format="free" }}',
        f'transform{{ function={"log" if log else "none"} }}',
        f'regression{{ user=({" ".join(exog.columns)})', f'  usertype=({types})',
        f'  start={sy}.{sm:02d} file="{tag}_x.dat" format="free"', '  save=(hol td) }',
        f'arima{{ model={arima} }}' if arima else 'automdl{ }',
        'outlier{ types=(ao ls) }', 'estimate{ }', f'forecast{{ maxlead={maxlead} }}', 'check{ }',
        'x11{ seasonalma=msr save=(d10 d11 d12 d13) print=(d8 f2 f3) }',
        'spectrum{ print=(specorig specsa specirr) }', ""])
    with open(base + ".spc", "w") as f: f.write(spc)
    r = subprocess.run([X13, tag], cwd=XWORK, capture_output=True, text=True)
    if arima is None and "singular" in (r.stdout + r.stderr):
        return x13_run(tag, y, exog, log, maxlead, arima="(0 1 1)(0 1 1)")
    out = open(base + ".out", encoding="latin-1").read()
    errs = [l.strip() for l in (r.stdout + r.stderr).splitlines() if "ERROR" in l]
    assert not errs, (tag, errs[:2])
    g = lambda p, i=1: (lambda m: m.group(i) if m else None)(re.search(p, out, re.M))
    fl = lambda v: float(v) if v is not None else np.nan
    res = {"tag": tag, "arima_fixed": arima, "arima": (g(r"ARIMA Model:\s*(\([^)]*\)\s*\([^)]*\))") or "").replace(" ", ""),
           "aicc": fl(g(r"AICC \(F-corrected-AIC\)\s+([-\d.]+)")),
           "coefs": {m.group(1): (float(m.group(2)), float(m.group(3)), float(m.group(4)))
                     for m in re.finditer(r"^\s+(ln_wd|seol_\w+|chu_\w+)\s+([-\d.]+)\s+([\d.]+)\s+([-\d.]+)\s*$", out, re.M)},
           "outliers": re.findall(r"^\s+((?:AO|LS|TC)\d{4}\.\w{3})\s+([-\d.]+)\s+[\d.]+\s+([-\d.]+)", out, re.M),
           "d8_stable": fl(g(r"F-test for stable seasonality from Table D 8\.\s*:\s*([\d.]+)")),
           "d8_stable_p": fl(g(r"F-test for stable seasonality from Table D 8\.\s*:\s*[\d.]+\s+([\d.]+)%")),
           "d8_moving": fl(g(r"F-test for moving seasonality from Table D 8\.\s*:\s*([\d.]+)")),
           "d8_moving_p": fl(g(r"F-test for moving seasonality from Table D 8\.\s*:\s*[\d.]+\s+([\d.]+)%")),
           "m7": fl(g(r"M7\s*=\s*([\d.]+)")), "q": fl(g(r"\*\*\* Q \(without M2\) =\s*([\d.]+)")),
           "q_all": fl(g(r"\*\*\* (?:CONDITIONALLY )?(?:ACCEPTED|REJECTED) \*\*\* at the level\s+([\d.]+)")),
           "qs_orig_p": fl(g(r"QS statistic for seasonality:\n(?:.*\n){0,8}?\s*Original Series \(EV adj\)\s+[\d.]+\s+\(P-Value =\s+([\d.]+)\)")),
           "qs_sa_p": fl(g(r"QS statistic for seasonality:\n(?:.*\n){0,8}?\s*Seasonally Adjusted Series \(EV adj\)\s+[\d.]+\s+\(P-Value =\s+([\d.]+)\)")),
           "td_peak": "residual trading day peaks" in out, "seas_peak": "residual seasonal peaks" in out}
    def table(ext):
        t = pd.read_csv(base + "." + ext, sep=r"\s+", skiprows=2, header=None, names=["date","v"], engine="python")
        return pd.Series(t.v.values, index=t.date.astype(int))
    for ext in ("d10","d11","d12","d13","td","hol"): res[ext] = table(ext)
    return res

X13R, SA = {}, {}
for i, s in enumerate(ORDER):
    y = W[s].dropna(); ex = Xcal.loc[y.index.min():, CAL_VARS]
    r = x13_run(f"s{i:02d}", y, ex); X13R[s] = r
    ln_y = np.log(y)
    ln_c = ln_y - np.log(r["td"].reindex(y.index)) - np.log(r["hol"].reindex(y.index))   # 달력조정
    sa_x = np.log(r["d11"].reindex(y.index)); seas_x = np.log(r["d10"].reindex(y.index))
    stl = STL(ln_c.values, period=12, robust=True, seasonal=13).fit()
    SA[s] = pd.DataFrame({"ln_y": ln_y, "ln_c": ln_c, "sa_B": sa_x, "seas_B": seas_x,
                          "sa_A": pd.Series(ln_c.values - stl.seasonal, index=y.index), "seas_A": pd.Series(stl.seasonal, index=y.index),
                          "trend_x13": np.log(r["d12"].reindex(y.index))})
    assert SA[s].isna().sum().sum() == 0, s

rows = []
for s in ORDER:
    r, d = X13R[s], SA[s]; c = r["coefs"]
    diff = (d.sa_A - d.sa_B) * 100
    rows.append({"계열": s, "시작": int(d.index.min()), "n": len(d), "ARIMA": r["arima"] + ("*" if r["arima_fixed"] else ""), "AICc": r["aicc"],
                 "β_D": c["ln_wd"][0], "t": c["ln_wd"][2], "설pre": c["seol_pre"][0], "설dur": c["seol_dur"][0], "설post": c["seol_post"][0],
                 "추석pre": c["chu_pre"][0], "추석dur": c["chu_dur"][0], "추석post": c["chu_post"][0],
                 "설Θ": c["seol_pre"][0] + c["seol_dur"][0] + c["seol_post"][0], "추석Θ": c["chu_pre"][0] + c["chu_dur"][0] + c["chu_post"][0],
                 "이상치 수": len(r["outliers"]), "이상치": " ".join(o[0] for o in r["outliers"]),
                 "D8 F안정": r["d8_stable"], "D8 F이동": r["d8_moving"], "M7": r["m7"], "Q": r["q"], "QS원 p": r["qs_orig_p"], "QS조정 p": r["qs_sa_p"],
                 "TD피크": r["td_peak"], "계절피크": r["seas_peak"],
                 "RMS차(%)": np.sqrt((diff**2).mean()), "최대차(%)": diff.abs().max(), "Δ상관": d.sa_A.diff().corr(d.sa_B.diff()),
                 "계절진폭B 초기(%)": (d.seas_B.iloc[:36].max() - d.seas_B.iloc[:36].min())*100,
                 "계절진폭B 말기(%)": (d.seas_B.iloc[-36:].max() - d.seas_B.iloc[-36:].min())*100,
                 "계절진폭A 초기(%)": (d.seas_A.iloc[:36].max() - d.seas_A.iloc[:36].min())*100,
                 "계절진폭A 말기(%)": (d.seas_A.iloc[-36:].max() - d.seas_A.iloc[-36:].min())*100})
TAB_SA = pd.DataFrame(rows).set_index("계열")
NO_SA = set(TAB_SA.index[TAB_SA["M7"] >= 1])
TAB_SA["계절조정"] = ["안 씀(M7≥1)" if s in NO_SA else "X-13" for s in TAB_SA.index]
TAB_SA.to_csv(os.path.join(OUT, "tab_sa_compare.csv"), encoding="utf-8-sig")
long = pd.concat([d.assign(series=s) for s, d in SA.items()]).reset_index().rename(columns={"index":"yyyymm"})
long.to_csv(os.path.join(CACHE, "series_sa.csv"), index=False, encoding="utf-8-sig")
print("계절조정 안 쓰는 계열(M7≥1):", sorted(NO_SA))
print(TAB_SA[["ARIMA","β_D","t","설dur","추석dur","이상치 수","D8 F안정","D8 F이동","M7","Q","QS원 p","QS조정 p","RMS차(%)","계절진폭B 초기(%)","계절진폭B 말기(%)"]].round(3).to_string())

계절조정 안 쓰는 계열(M7≥1): ['선박']
                 ARIMA    β_D      t   설dur  추석dur  이상치 수  D8 F안정  D8 F이동     M7     Q  QS원 p  QS조정 p  RMS차(%)  계절진폭B 초기(%)  계절진폭B 말기(%)
계열                                                                                                                                      
총수출         (112)(011)  0.364  10.54 -0.041 -0.046      2  60.749   2.564  0.348  0.42  0.000   1.000    1.422       23.490       14.279
반도체 제외      (110)(011)  0.435  12.47 -0.029 -0.051      3  58.142   3.362  0.383  0.45  0.000   1.000    1.040       27.458       12.521
반도체         (311)(011)  0.100   2.06 -0.020 -0.011      1  32.576   2.751  0.484  0.42  0.000   1.000    4.509       17.870       28.985
화공품         (311)(011)  0.455  12.94 -0.004 -0.023      2  22.198   1.397  0.502  0.63  0.000   1.000    1.023       15.357       12.741
승용차         (111)(011)  0.519   5.46 -0.050 -0.074      8  47.277   3.367  0.425  0.59  0.000   1.000    3.211       51.328       23.609
석유제품        (0

In [5]:
# 사양 B(센서스 표준: 요일 6계수 + 평일 공휴일 수 + 명절 창)를 총수출에서만 견준다. II.3절이 이 차이를 적는다.
days = pd.date_range(f"{YM_START//100}-{YM_START%100:02d}-01", f"{YM_LEAD//100}-{YM_LEAD%100:02d}-28", freq="D")
dd = pd.DataFrame({"d": days}); dd["ym"] = dd.d.dt.year*100 + dd.d.dt.month
dd["n_hol"] = ((dd.d.dt.weekday < 5) & dd.d.dt.date.isin(hol)).astype(float)
NH = dd.groupby("ym").n_hol.sum().to_frame()
XB = NH.join(H).fillna(0.0).loc[YM_START:YM_LEAD].copy(); XB["month"] = XB.index % 100
for c in ["n_hol"] + CAL_VARS[1:]:
    mm = XB.loc[:YM_END].groupby("month")[c].mean(); XB[c] = XB[c] - XB["month"].map(mm)

def x13_run_B(tag, y, exog):
    """사양 B: X-13 내장 요일 설명변수(td) + 사용자 정의 설명변수(공휴일 수·명절 창)."""
    base = os.path.join(XWORK, tag); sy, sm = divmod(int(y.index.min()), 100)
    with open(base + ".dat", "w") as f:
        for v in y.values: f.write(f"{v:.6f}\n")
    with open(base + "_x.dat", "w") as f:
        for row in exog.values: f.write(" ".join(f"{v:.8f}" for v in row) + "\n")
    types = " ".join(["holiday"] * exog.shape[1])
    spc = "\n".join([f'series{{ title="{tag}" start={sy}.{sm:02d} period=12 file="{tag}.dat" format="free" }}', 'transform{ function=log }',
        f'regression{{ variables=(td) user=({" ".join(exog.columns)})', f'  usertype=({types})', f'  start={sy}.{sm:02d} file="{tag}_x.dat" format="free" }}',
        'automdl{ }', 'outlier{ types=(ao ls) }', 'estimate{ }', f'forecast{{ maxlead={LEAD} }}', 'x11{ seasonalma=msr }', ""])
    with open(base + ".spc", "w") as f: f.write(spc)
    r = subprocess.run([X13, tag], cwd=XWORK, capture_output=True, text=True)
    out = open(base + ".out", encoding="latin-1").read()
    g = lambda p: (lambda m: m.group(1) if m else None)(re.search(p, out, re.M))
    fl = lambda v: float(v) if v is not None else np.nan
    td = {m.group(1): (float(m.group(2)), float(m.group(3))) for m in re.finditer(r"^\s+(Mon|Tue|Wed|Thu|Fri|Sat)\s+([-\d.]+)\s+[\d.]+\s+([-\d.]+)", out, re.M)}
    us = {m.group(1): (float(m.group(2)), float(m.group(3))) for m in re.finditer(r"^\s+(n_hol|seol_\w+|chu_\w+)\s+([-\d.]+)\s+[\d.]+\s+([-\d.]+)", out, re.M)}
    return {"aicc": fl(g(r"AICC \(F-corrected-AIC\)\s+([-\d.]+)")), "arima": (g(r"ARIMA Model:\s*(\([^)]*\)\s*\([^)]*\))") or "").replace(" ", ""),
            "td": td, "user": us, "td_peak": "residual trading day peaks" in out}

yT = W["총수출"]; SPEC_B = x13_run_B("sB_total", yT, XB.loc[yT.index.min():, ["n_hol"] + CAL_VARS[1:]])
SPEC_A = X13R["총수출"]
print(f"총수출  A: AICc {SPEC_A['aicc']:.1f} ARIMA {SPEC_A['arima']}  |  B: AICc {SPEC_B['aicc']:.1f} ARIMA {SPEC_B['arima']}  | ΔAICc(A-B) {SPEC_A['aicc']-SPEC_B['aicc']:.1f}")
print("B 요일 계수(t):", {k: (round(v[0],4), round(v[1],2)) for k, v in SPEC_B["td"].items()})
print("B 공휴일 수 계수(t):", SPEC_B["user"]["n_hol"], "| A ln_wd 하루 환산:", round(SPEC_A["coefs"]["ln_wd"][0] / 21, 4))
print("TD 피크 경고: A", SPEC_A["td_peak"], "B", SPEC_B["td_peak"])

총수출  A: AICc 16500.9 ARIMA (112)(011)  |  B: AICc 16490.9 ARIMA (112)(011)  | ΔAICc(A-B) 10.0
B 요일 계수(t): {'Mon': (0.0006, 0.15), 'Tue': (0.0091, 2.26), 'Wed': (0.0022, 0.53), 'Thu': (0.0062, 1.55), 'Fri': (0.0075, 1.87), 'Sat': (-0.0069, -1.7)}
B 공휴일 수 계수(t): (-0.0153, -5.47) | A ln_wd 하루 환산: 0.0174
TD 피크 경고: A False B False


In [6]:
# 계절조정 결과의 요약 검증: Δ12는 조정 전후가 거의 같아야 하고, 월간 변화의 표준편차는 조정 후가 작아야 한다.
chk_rows = []
for s in ORDER:
    d = SA[s]
    chk_rows.append({"계열": s,
        "Δ12 원 vs A 상관": d.ln_y.diff(12).corr(d.sa_A.diff(12)), "Δ12 원 vs B 상관": d.ln_y.diff(12).corr(d.sa_B.diff(12)),
        "sd Δ1 원(%)": d.ln_y.diff().std()*100, "sd Δ1 A(%)": d.sa_A.diff().std()*100, "sd Δ1 B(%)": d.sa_B.diff().std()*100})
CHK = pd.DataFrame(chk_rows).set_index("계열")
print(CHK.round(3).to_string())
AGG = ["총수출","반도체 제외","총수입","에너지 제외 수입"]
assert (CHK.loc[AGG, "sd Δ1 B(%)"] < CHK.loc[AGG, "sd Δ1 원(%)"]).all()
assert (CHK["Δ12 원 vs B 상관"] > 0.9).all()
NOISIER = CHK.index[CHK["sd Δ1 B(%)"] >= CHK["sd Δ1 원(%)"]].tolist()
print("X-13 조정 뒤 월간 변동이 안 줄어든 계열:", NOISIER)

           Δ12 원 vs A 상관  Δ12 원 vs B 상관  sd Δ1 원(%)  sd Δ1 A(%)  sd Δ1 B(%)
계열                                                                         
총수출                0.972          0.972       8.694       4.513       4.119
반도체 제외             0.953          0.952       9.231       4.649       4.460
반도체                0.998          0.999      10.445       9.189       6.750
화공품                0.977          0.977       7.593       4.761       4.574
승용차                0.955          0.955      21.678      14.035      14.186
석유제품               0.995          0.994      17.756      15.366      15.529
일반기계               0.950          0.949      13.703       7.018       6.862
철강제품               0.970          0.970      10.125       7.188       6.895
선박                 0.986          0.984      51.104      41.371      39.256
자동차부품              0.969          0.970      15.050      10.940       9.951
컴퓨터주변기기            0.996          0.996      13.460      11.659       9.943
무선통신기기      

## §4. 트렌드와 순환의 분해 (2단계)

계절조정 계열(X-13, M7≥1인 선박은 달력조정 계열)에 세 방법을 적용해 트렌드 $T_t$를 얻고 트렌드 성장률을 $\Delta_{12}\ln T_t$로 통일한다.

- **HP(본방법)**: $\lambda = 129{,}600$.
- **UCM(강건성)**: 평활 트렌드(수준 잡음 없이 기울기만 확률적) + 확률적 감쇠 순환(주기 1.5~10년) + 불규칙, 상태공간 MLE, 평활 추정치. 세 최적화 경로(lbfgs, nm+lbfgs, powell) 가운데 우도가 가장 큰 해를 고른다. 우도면에 봉우리가 여럿이라 계열에 따라 트렌드와 순환의 배분이 정해지지 않는다(반도체는 거의 직선 트렌드로 간다). 총수출·반도체 제외에서는 세 경로가 같은 해(주기 45개월)에 이르고 HP와 상관 0.98~0.99다. 이 불안정 때문에 본방법을 HP로 둔다.
- **STL 트렌드**: §3의 STL 트렌드 성분(트렌드창은 기본값 23개월이라 셋 중 가장 거칠다). 참고용이며 논문에는 싣지 않는다.

세 방법의 트렌드 성장률이 서로 얼마나 맞는지(상관), 그리고 부호가 바뀌는 시점이 같은지를 본다.

In [7]:
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.filters.hp_filter import hpfilter

# NO_SA는 §3에서 X-13의 M7 기준으로 정했다.
def base_series(s):
    d = SA[s]
    return d["ln_c"] if s in NO_SA else d["sa_B"]

def fit_ucm(y):
    mod = UnobservedComponents(y.values, level="smooth trend", cycle=True, stochastic_cycle=True,
                               damped_cycle=True, irregular=True, cycle_period_bounds=(18, 120))
    # 우도면이 여러 봉우리를 가져 시작점에 따라 다른 해에 앉는다(powell은 주기 120개월의 나쁜 해로 자주 간다).
    # 세 경로를 다 돌리고 우도가 가장 큰 해를 고른다.
    cands = []
    for how in ("lbfgs", "nm+lbfgs", "powell"):
        try:
            if how == "lbfgs": r = mod.fit(disp=0, maxiter=1000)
            elif how == "powell": r = mod.fit(method="powell", disp=0, maxiter=3000)
            else:
                r0 = mod.fit(method="nm", disp=0, maxiter=5000); r = mod.fit(start_params=r0.params, disp=0, maxiter=1000)
            cands.append((r.llf, how, r))
        except Exception:
            pass
    cands.sort(key=lambda t: -t[0]); res = cands[0][2]; res._how = cands[0][1]
    return res

TR = {}
for s in ORDER:
    y = base_series(s)
    r = fit_ucm(y)
    lvl = pd.Series(r.level["smoothed"], index=y.index)
    slope = pd.Series(r.trend["smoothed"], index=y.index)
    cyc = pd.Series(r.cycle["smoothed"], index=y.index)
    hp_c, hp_t = hpfilter(y.values, lamb=129600)
    stl_t = pd.Series(STL(y.values, period=12, robust=True, seasonal=13).fit().trend, index=y.index)
    hp_t = pd.Series(hp_t, index=y.index)
    TR[s] = pd.DataFrame({"y": y, "ucm_level": lvl, "ucm_slope": slope, "ucm_cycle": cyc,
                          "hp_trend": hp_t, "stl_trend": stl_t,
                          "g_ucm": lvl.diff(12), "g_hp": hp_t.diff(12), "g_stl": stl_t.diff(12), "g_sa": y.diff(12)})
    TR[s]["g_main"] = TR[s]["g_hp"]; TR[s]["trend_main"] = TR[s]["hp_trend"]     # 본방법 HP
    TR[s].attrs["ucm_period"] = float(2*np.pi / np.asarray(r.params)[list(r.param_names).index("frequency.cycle")])
    TR[s].attrs["ucm_converged"] = bool(r.mle_retvals.get("converged", True)); TR[s].attrs["ucm_how"] = r._how

rows = []
for s in ORDER:
    d = TR[s].dropna()
    rows.append({"계열": s, "UCM 수렴": TR[s].attrs["ucm_converged"], "UCM 경로": TR[s].attrs["ucm_how"], "UCM 순환주기(월)": TR[s].attrs["ucm_period"],
                 "corr(UCM,HP)": d.g_ucm.corr(d.g_hp), "corr(UCM,STL)": d.g_ucm.corr(d.g_stl), "corr(HP,STL)": d.g_hp.corr(d.g_stl),
                 "sd g_UCM(%)": d.g_ucm.std()*100, "sd g_HP(%)": d.g_hp.std()*100, "sd g_STL(%)": d.g_stl.std()*100,
                 "sd g_sa(%)": d.g_sa.std()*100,
                 "부호전환 UCM": int((np.sign(d.g_ucm).diff().abs() > 0).sum()),
                 "부호전환 HP": int((np.sign(d.g_hp).diff().abs() > 0).sum()),
                 "부호전환 STL": int((np.sign(d.g_stl).diff().abs() > 0).sum())})
TAB_TR = pd.DataFrame(rows).set_index("계열")
TAB_TR.to_csv(os.path.join(OUT, "tab_trend_methods.csv"), encoding="utf-8-sig")
pd.concat([d.assign(series=s) for s, d in TR.items()]).reset_index().rename(columns={"index": "yyyymm"}) \
  .to_csv(os.path.join(CACHE, "series_trend.csv"), index=False, encoding="utf-8-sig")
print(TAB_TR.round(2).to_string())
print("UCM 미수렴:", TAB_TR.index[~TAB_TR["UCM 수렴"]].tolist(), "| UCM 경로:", TAB_TR["UCM 경로"].value_counts().to_dict())

           UCM 수렴    UCM 경로  UCM 순환주기(월)  corr(UCM,HP)  corr(UCM,STL)  corr(HP,STL)  sd g_UCM(%)  sd g_HP(%)  sd g_STL(%)  sd g_sa(%)  부호전환 UCM  부호전환 HP  부호전환 STL
계열                                                                                                                                                                
총수출          True  nm+lbfgs        44.66          0.98           0.56          0.54         4.72        4.40        10.73       14.82         2        2        14
반도체 제외       True  nm+lbfgs        45.05          0.99           0.61          0.60         5.08        4.93         9.34       13.47         2        2        16
반도체          True    powell        50.09          0.58           0.23          0.63         0.83        5.23        20.40       31.42         0        0        14
화공품          True  nm+lbfgs        50.98          0.99           0.56          0.59         5.40        5.77        12.38       17.42         1        1        13
승용차          True    p

## §5. 구조변화 검정 (2단계)

대상은 계열별 $\Delta_{12}\ln y_t$(계절조정 뒤, M7 ≥ 1인 선박은 달력조정 계열)이고 모형은 평균 이동이다. 최소 구간 24개월, 단절 최대 5개.
동적 계획법으로 단절 수마다 최소 잔차제곱합 분할을 찾고, 단절 수는 LWZ(Liu-Wu-Zidek)로 고른다. BIC도 함께 적되 $\Delta_{12}$ 계열은
12개월 겹침으로 자기상관이 강해 BIC가 단절을 많이 고르는 쪽으로 치우친다. 단절 하나의 검정은 Andrews sup-$F$(절사 15%, HAC 12)이고
임계값은 Andrews(1993) $p=1$의 7.17(10%)·8.85(5%)·12.35(1%)다. 자료 끝에서 24개월 안에 생긴 단절은 정의상 잡히지 않는다(2024년 8월 이후).

In [8]:
H_MIN, M_MAX = 24, 5

def bp_dp(y, h=H_MIN, mmax=M_MAX):
    """Bai-Perron 평균이동 다중 단절: 동적 계획법. 단절 위치는 구간 마지막 관측의 0-based 인덱스."""
    y = np.asarray(y, float); T = len(y)
    cs = np.concatenate([[0.0], np.cumsum(y)]); cs2 = np.concatenate([[0.0], np.cumsum(y**2)])
    INF = np.inf
    C = np.full((T, T), INF)
    for i in range(T):
        j = np.arange(i + h - 1, T)
        n = j - i + 1; s1 = cs[j+1] - cs[i]; s2 = cs2[j+1] - cs2[i]
        C[i, j] = s2 - s1*s1/n
    F = np.full((mmax + 1, T), INF); arg = np.full((mmax + 1, T), -1, dtype=int)
    F[0, :] = C[0, :]
    for m in range(1, mmax + 1):
        for t in range(T):
            lo, hi = m*h - 1, t - h + 1
            if hi <= lo: continue
            cand = F[m-1, lo:hi] + C[lo+1:hi+1, t]
            k = int(np.argmin(cand))
            F[m, t], arg[m, t] = cand[k], lo + k
    out = {}
    for m in range(mmax + 1):
        if not np.isfinite(F[m, T-1]): break
        br, t, k = [], T - 1, m
        while k > 0:
            s_ = arg[k, t]; br.append(s_); t, k = s_, k - 1
        out[m] = (F[m, T-1], sorted(br))
    return out

def choose_m(out, T):
    rows = []
    for m, (S_, br) in out.items():
        p = 2*m + 1
        bic = np.log(S_/T) + p*np.log(T)/T
        lwz = np.log(S_/(T-p)) + p*0.299*np.log(T)**2.1/T
        rows.append((m, S_, bic, lwz))
    d = pd.DataFrame(rows, columns=["m","ssr","bic","lwz"]).set_index("m")
    return int(d.bic.idxmin()), int(d.lwz.idxmin()), d

def supF(y, trim=0.15, hac=12):
    y = np.asarray(y, float); T = len(y)
    lo, hi = int(np.floor(trim*T)), int(np.ceil((1-trim)*T))
    best = (-np.inf, None)
    for k in range(lo, hi):
        D = (np.arange(T) >= k).astype(float)
        r = sm.OLS(y, sm.add_constant(D)).fit(cov_type="HAC", cov_kwds={"maxlags": hac})
        w = r.tvalues[1]**2
        if w > best[0]: best = (w, k)
    return best

def ym_add(ym, k):
    y, m = divmod(int(ym), 100); m = m - 1 + k
    return (y + m // 12) * 100 + m % 12 + 1

BP = {}; rows = []
for s in ORDER:
    g = TR[s].g_sa.dropna()
    out = bp_dp(g.values); m_bic, m_lwz, crit = choose_m(out, len(g))
    Wst, k = supF(g.values)
    first_of = lambda b: ym_add(g.index[b], 1)          # 단절 다음 달 = 새 국면 첫 달
    br_lwz = [first_of(b) for b in out[m_lwz][1]]; br_bic = [first_of(b) for b in out[m_bic][1]]
    segs = []; bounds = [-1] + out[m_lwz][1] + [len(g) - 1]
    for a, b in zip(bounds[:-1], bounds[1:]):
        seg = g.iloc[a+1:b+1]; segs.append((int(g.index[a+1]), int(g.index[b]), seg.mean()*100))
    BP[s] = {"m_lwz": m_lwz, "m_bic": m_bic, "br_lwz": br_lwz, "br_bic": br_bic, "supF": Wst,
             "supF_at": int(g.index[k]), "segments": segs, "crit": crit}
    rows.append({"계열": s, "supF": Wst, "supF 시점": int(g.index[k]), "m(LWZ)": m_lwz, "m(BIC)": m_bic,
                 "단절(LWZ, 새 국면 첫 달)": br_lwz, "단절(BIC)": br_bic})
TAB_BP = pd.DataFrame(rows).set_index("계열")
TAB_BP.to_csv(os.path.join(OUT, "tab_breaks.csv"), encoding="utf-8-sig")
pd.set_option("display.max_colwidth", 80)
print(TAB_BP.round(1).to_string())

           supF  supF 시점  m(LWZ)  m(BIC)                         단절(LWZ, 새 국면 첫 달)                                   단절(BIC)
계열                                                                                                                          
총수출         2.2   201201       0       5                                        []  [200207, 200712, 200912, 201112, 202011]
반도체 제외      5.8   201201       2       5                          [200207, 200811]  [200207, 200712, 200912, 201201, 202012]
반도체         2.3   200207       1       5                                  [202401]  [200206, 200709, 200909, 201812, 202401]
화공품         6.7   202112       4       5          [200911, 201111, 202009, 202209]  [200711, 200911, 201111, 202009, 202209]
승용차         4.5   200709       3       5                  [200712, 200912, 201204]  [200712, 200912, 201205, 202101, 202401]
석유제품        3.4   201203       0       5                                        []  [201410, 201610, 201902, 202103, 202303]


In [9]:
# 국면표: LWZ 단절로 나눈 구간과 구간 평균 성장률(Δ12 원계열, HP 트렌드 성장률)
def phase_table(s):
    g = TR[s]; rows = []
    for a, b, mean_raw in BP[s]["segments"]:
        seg = g.loc[a:b]
        rows.append({"계열": s, "시작": a, "끝": b, "개월": len(seg), "Δ12 평균(%)": mean_raw,
                     "HP 트렌드 성장 평균(%)": seg.g_main.mean()*100, "Δ12 표준편차(%)": seg.g_sa.std()*100})
    return pd.DataFrame(rows)
PH = pd.concat([phase_table(s) for s in ORDER], ignore_index=True)
PH.to_csv(os.path.join(OUT, "tab_phases.csv"), index=False, encoding="utf-8-sig")
for s in ["총수출", "반도체 제외", "반도체", "총수입"]:
    print(f"\n[{s}]  supF={BP[s]['supF']:.1f} @ {BP[s]['supF_at']}  m(LWZ)={BP[s]['m_lwz']}  m(BIC)={BP[s]['m_bic']}")
    print(PH[PH.계열 == s].drop(columns="계열").round(1).to_string(index=False))


[총수출]  supF=2.2 @ 201201  m(LWZ)=0  m(BIC)=5
    시작      끝  개월  Δ12 평균(%)  HP 트렌드 성장 평균(%)  Δ12 표준편차(%)
199601 202607 367        6.5              6.2         14.8

[반도체 제외]  supF=5.8 @ 201201  m(LWZ)=2  m(BIC)=5
    시작      끝  개월  Δ12 평균(%)  HP 트렌드 성장 평균(%)  Δ12 표준편차(%)
199601 200206  78        3.5              5.3         11.7
200207 200810  76       17.1             13.1          7.2
200811 202607 213        2.2              2.9         13.6

[반도체]  supF=2.3 @ 200207  m(LWZ)=1  m(BIC)=5
    시작      끝  개월  Δ12 평균(%)  HP 트렌드 성장 평균(%)  Δ12 표준편차(%)
199601 202312 336        6.2              7.5         29.6
202401 202607  31       42.8             21.0         31.3

[총수입]  supF=1.5 @ 201203  m(LWZ)=0  m(BIC)=4
    시작      끝  개월  Δ12 평균(%)  HP 트렌드 성장 평균(%)  Δ12 표준편차(%)
199801 202607 343        5.5              6.2         19.9


## §6. 그림 1·3 (2단계)

그림 1은 총수출·반도체·반도체 제외의 HP 트렌드 성장률 월별 계열(1996.01~), 그림 3은 총수출 계절조정 로그 수준과 HP 트렌드의 겹침이다. 총수출에는 LWZ 단절이 없어 반도체 제외의 두 단절을 점선으로 표시한다.
흑백, Malgun Gothic, 300dpi, 위·오른쪽 축선 제거, 격자 0.87.

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False, "figure.dpi": 100})

def style(ax):
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.grid(True, color="0.87", linewidth=0.8); ax.set_axisbelow(True)
def tdate(idx): return pd.to_datetime(pd.Index(idx).astype(int).astype(str), format="%Y%m")

fig, ax = plt.subplots(figsize=(10, 4.2))
for s, ls, lw, col in [("총수출", "-", 1.8, "black"), ("반도체 제외", "--", 1.4, "black"), ("반도체", "-", 1.0, "0.55")]:
    d = TR[s].g_main.dropna()
    ax.plot(tdate(d.index), d.values*100, ls, lw=lw, color=col, label=s)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("HP 트렌드 성장률 (전년 동월 대비, %)"); ax.legend(frameon=False, ncol=3, loc="upper left")
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig1_trend_growth.png"), dpi=300); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4.2))
d = TR["총수출"]
ax.plot(tdate(d.index), d.y.values, color="0.6", lw=0.8, label="계절조정 로그 수준")
ax.plot(tdate(d.index), d.trend_main.values, color="black", lw=1.6, label="HP 트렌드")
# 총수출에는 LWZ 단절이 없다. 반도체 제외의 두 단절을 점선으로 표시한다.
for b in BP["반도체 제외"]["br_lwz"]:
    ax.axvline(tdate([b])[0], color="black", lw=0.8, ls=":")
ax.text(0.99, 0.03, "점선: 반도체 제외 수출의 단절 (%s)" % ", ".join(f"{b//100}.{b%100:02d}" for b in BP["반도체 제외"]["br_lwz"]),
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="0.3")
ax.set_ylabel("ln(총수출, 달러)"); ax.legend(frameon=False, loc="upper left")
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig3_level_trend_breaks.png"), dpi=300); plt.close(fig)
print("그림 저장:", sorted(os.listdir(IMG)))

그림 저장: ['fig1_trend_growth.png', 'fig2_rolling_corr.png', 'fig3_level_trend_breaks.png', 'fig_b1_exports_ip.png', 'fig_b2_semi_ip.png', 'fig_b3_rolling_corr.png', 'fig_c1_trend_growth_lambda.png', 'fig_c2_realtime_total.png', 'fig_c3_realtime_exsemi.png']


## §7. 구성의 변화 (3단계)

국면은 2단계 결과에 따라 사전 후보 가운데 다섯 경계(2001·2008·2016·2020·2024)로 여섯을 둔다. 반도체 제외의 두 단절(2002년 7월, 2008년 11월)이 앞의 둘에 대응하고,
2016년은 총수출 트렌드 성장률의 저점, 2020년은 코로나, 2024년은 반도체 단절이다. 연 단위 비중은 대분류 넷과 품목군 열다섯(+나머지)에서 잰다.
집중도는 부호 153개의 허핀달 지수와 상위 3개 부호 비중. 기여도는 국면 끝점 사이의 증가율을 품목군별로 나눈 것이며 마지막 국면의 끝점은 2026년 7월까지 12개월 합이다.

In [11]:
PHASES = [("P1", 1995, 2000), ("P2", 2001, 2007), ("P3", 2008, 2015), ("P4", 2016, 2019), ("P5", 2020, 2023), ("P6", 2024, 2026)]
PH_LABEL = {p: f"{a}~{b if b < 2026 else '2026.07'}" for p, a, b in PHASES}

# 연도별 부호 단위 수출 (금액·중량). 마지막 '연도' 2026은 2025.08~2026.07의 12개월 합으로 둔다(끝점).
raw = con.execute("SELECT yyyymm, temper_cd, dlr, wgt FROM fact_temper WHERE imexp='수출'").df()
raw["year"] = raw.yyyymm // 100
ann = raw[raw.year <= 2025].groupby(["year","temper_cd"])[["dlr","wgt"]].sum().reset_index()
ttm = raw[(raw.yyyymm >= 202508) & (raw.yyyymm <= 202607)].groupby("temper_cd")[["dlr","wgt"]].sum().reset_index().assign(year=2026)
ann = pd.concat([ann, ttm], ignore_index=True).merge(MAP[["temper_cd","group15","x1"]], on="temper_cd", how="left")
assert ann.group15.notna().all()
ann["x1s"] = ann.x1.str.replace(r"^\d\.\s*", "", regex=True)

# 표: 대분류 비중(연도별)
tot_y = ann.groupby("year").dlr.sum()
sh_x1 = (ann.groupby(["year","x1s"]).dlr.sum().unstack() .div(tot_y, axis=0) * 100)
sh_g  = (ann.groupby(["year","group15"]).dlr.sum().unstack().div(tot_y, axis=0) * 100)[EXP15 + ["나머지"]]
hh = ann.groupby("year").apply(lambda d: ((d.dlr / d.dlr.sum())**2).sum() * 10000).rename("HHI")
top3 = ann.groupby("year").apply(lambda d: d.dlr.nlargest(3).sum() / d.dlr.sum() * 100).rename("top3(%)")
semi = sh_g["반도체"].rename("반도체(%)")
TAB_SHARE = pd.concat([sh_x1.round(1), semi.round(1), hh.round(0), top3.round(1)], axis=1)
TAB_SHARE.to_csv(os.path.join(OUT, "tab_share_x1.csv"), encoding="utf-8-sig")
sh_g.round(1).to_csv(os.path.join(OUT, "tab_share_group.csv"), encoding="utf-8-sig")
print(TAB_SHARE.loc[[1995, 2000, 2005, 2008, 2010, 2015, 2016, 2019, 2020, 2023, 2024, 2025, 2026]].to_string())
print()
print(sh_g.loc[[1995, 2000, 2007, 2015, 2019, 2023, 2026]].round(1).T.to_string())

      경공업품  식료 및 직접소비재  원료 및 연료  중화학 공업품  반도체(%)     HHI  top3(%)
year                                                             
1995  24.3         2.4      3.7     69.6    14.1   343.0     23.1
2000  17.6         1.6      6.7     74.1    15.1   406.0     27.4
2005   9.3         1.1      6.6     83.1    11.3   395.0     25.1
2008   7.0         1.0     10.5     81.6     7.8   403.0     25.4
2010   7.0         1.1      8.3     83.6    11.0   388.0     23.4
2015   6.7         1.3      7.5     84.5    12.0   368.0     22.8
2016   7.2         1.5      6.7     84.7    12.7   370.0     22.3
2019   6.3         1.5      9.0     83.2    17.8   420.0     27.2
2020   6.3         1.7      6.3     85.7    19.9   423.0     27.0
2023   5.3         1.7      9.9     83.1    15.9   422.0     26.6
2024   5.0         1.7      8.9     84.4    21.0   478.0     29.8
2025   5.0         1.7      7.9     85.4    24.7   563.0     32.6
2026   4.5         1.4      7.4     86.7    35.3  1027.0     41.0

year     

In [12]:
# 기여도 분해: 국면 끝점(직전 국면의 마지막 해 → 이 국면의 마지막 해). 연율화는 햇수로 나눈다(가법성 유지).
V = ann.groupby(["year","group15"]).dlr.sum().unstack()[EXP15 + ["나머지"]]
V["총수출"] = V.sum(axis=1)
def yrs(a, b): return (b - a) if b < 2026 else (2025 + 7/12 - a)      # 2026 끝점은 2025.08~2026.07 합. 2023년 창과의 간격은 2.58년
rows = []
for p, a, b in PHASES:
    y0 = a - 1 if a > 1995 else 1995; y1 = b
    if y0 == y1: continue
    n = yrs(y0, y1)
    g = (V.loc[y1, "총수출"] / V.loc[y0, "총수출"] - 1) * 100
    r = {"국면": PH_LABEL[p], "기준": f"{y0}→{y1 if y1 < 2026 else '2026.07'}", "햇수": n,
         "총수출 증가율(%)": g, "연평균(%)": ((V.loc[y1, "총수출"] / V.loc[y0, "총수출"]) ** (1/n) - 1) * 100}
    for s in EXP15 + ["나머지"]:
        r[s] = (V.loc[y1, s] - V.loc[y0, s]) / V.loc[y0, "총수출"] * 100 / n    # 연평균 기여도(%p)
    rows.append(r)
TAB_CONTRIB = pd.DataFrame(rows).set_index("국면")
chk_sum = TAB_CONTRIB[EXP15 + ["나머지"]].sum(axis=1) * TAB_CONTRIB["햇수"]
assert np.allclose(chk_sum, TAB_CONTRIB["총수출 증가율(%)"]), "기여도 합이 증가율과 다르다"
TAB_CONTRIB.to_csv(os.path.join(OUT, "tab_contrib.csv"), encoding="utf-8-sig")
print(TAB_CONTRIB.round(2).T.to_string())

국면          1995~2000  2001~2007  2008~2015  2016~2019  2020~2023  2024~2026.07
기준          1995→2000  2000→2007  2007→2015  2015→2019  2019→2023  2023→2026.07
햇수                5.0        7.0        8.0        4.0        4.0          2.58
총수출 증가율(%)      37.75     115.65       41.8       2.94       16.6         43.78
연평균(%)           6.62       11.6       4.46       0.73       3.91         15.09
석유제품              1.1       1.23       0.27       0.42       0.51          0.19
화공품              0.72       1.89       0.59       0.45       0.77         -0.45
의약품              0.01       0.04       0.05       0.09       0.11          0.23
철강제품             0.22       1.68       0.33       0.13       0.26         -0.12
일반기계             0.26       1.55       0.55       0.19       0.26         -0.34
정밀기기             0.13       0.45       0.04       0.14       0.01         -0.02
제조장비             0.07       0.12       0.12       0.16      -0.04          0.21
가전제품             0.11      -0.26       0

## §8. 단가와 물량의 분해 (3단계)

부호 $i$의 단가 $p_{it} = V_{it}/W_{it}$, $\Delta\ln V = \Delta\ln p + \Delta\ln W$. 품목군은 부호별 $\Delta\ln p$를 두 끝점의 평균 금액 비중으로 가중한 Törnqvist 단가 변화로 합산하고
물량 변화는 잔차 $\Delta\ln V - \Delta\ln P$로 둔다. 끝점은 §7과 같고, 두 끝점 중 한쪽에서 중량이 0인 부호는 가중에서 뺀다. 단가는 가격이 아니라 단위중량당 가치다(계획서 VIII장).

In [13]:
def tornqvist(d0, d1):
    """d0·d1: temper_cd 인덱스, 열 dlr·wgt. 반환 (Δln V, Δln P, Δln Q) ×100(%)."""
    both = d0.join(d1, lsuffix="0", rsuffix="1", how="inner")
    both = both[(both.dlr0 > 0) & (both.dlr1 > 0) & (both.wgt0 > 0) & (both.wgt1 > 0)]
    w = 0.5 * (both.dlr0 / both.dlr0.sum() + both.dlr1 / both.dlr1.sum())
    dlp = (w * np.log((both.dlr1 / both.wgt1) / (both.dlr0 / both.wgt0))).sum()
    dlv = np.log(d1.dlr.sum() / d0.dlr.sum())
    return dlv*100, dlp*100, (dlv - dlp)*100

A = ann.set_index(["year","temper_cd"])
rows = []
for p, a, b in PHASES:
    y0 = a - 1 if a > 1995 else 1995; y1 = b
    n = yrs(y0, y1)
    for s in ["총수출", "반도체 제외"] + EXP15 + ["나머지"]:
        if s == "총수출": sel = ann.temper_cd.unique()
        elif s == "반도체 제외": sel = MAP.loc[MAP.group15 != "반도체", "temper_cd"].values
        else: sel = MAP.loc[MAP.group15 == s, "temper_cd"].values
        d0 = A.loc[y0].reindex(sel).dropna(); d1 = A.loc[y1].reindex(sel).dropna()
        dv, dp, dq = tornqvist(d0[["dlr","wgt"]], d1[["dlr","wgt"]])
        rows.append({"국면": PH_LABEL[p], "계열": s, "ΔlnV 연평균(%)": dv/n, "ΔlnP 연평균(%)": dp/n, "ΔlnQ 연평균(%)": dq/n})
TAB_PQ = pd.DataFrame(rows)
TAB_PQ.to_csv(os.path.join(OUT, "tab_price_quantity.csv"), index=False, encoding="utf-8-sig")
piv = TAB_PQ.pivot(index="계열", columns="국면", values=["ΔlnP 연평균(%)", "ΔlnQ 연평균(%)"]).reindex(["총수출","반도체 제외"] + EXP15 + ["나머지"])
print(piv.round(1).to_string())

        ΔlnP 연평균(%)                                                      ΔlnQ 연평균(%)                                                     
국면        1995~2000 2001~2007 2008~2015 2016~2019 2020~2023 2024~2026.07   1995~2000 2001~2007 2008~2015 2016~2019 2020~2023 2024~2026.07
계열                                                                                                                                       
총수출            -2.5       5.7       3.3       1.0       4.3         13.5         8.9       5.3       1.1      -0.2      -0.4          0.6
반도체 제외         -1.9       6.0       2.3      -0.3       5.6          3.6         8.0       5.7       1.9      -0.7      -1.2          0.3
석유제품            8.4      12.8      -3.0       4.0       7.9          2.1        18.5       0.8       6.6       2.1      -2.0          0.2
화공품            -6.1       7.6       0.3       0.9       6.2         -4.6        14.4       6.8       4.6       3.2      -0.3          0.8
의약품             2.0       6.9     

## §9. 변동성과 동조성 (3단계), 그림 2

품목군의 표준편차와 쌍 상관은 계절조정 계열의 $\Delta_{12}$로 잰다. 총수출 $\Delta_{12}$의 표준편차와 분산 몫 $\mathrm{cov}(c_i, g)/\mathrm{var}(g)$는 항등식을 지키려고 원계열 금액으로 잰다.
여기서 $c_{i,t} = (V_{i,t}-V_{i,t-12})/V_{\mathrm{tot},t-12}$는 항등식 $\sum_i c_{i,t} = g_t$를 만족하는 기여도다. 동조성은 반도체와 반도체 제외의 $\Delta_{12}$ 60개월 이동 상관(그림 2)과
국면별 품목군 쌍 상관의 평균이다.

In [14]:
Wm = S.pivot(index="yyyymm", columns="series", values="dlr")
G = Wm[EXP15 + ["나머지"]]; TOT = Wm["총수출"]
c = G.sub(G.shift(12)).div(TOT.shift(12), axis=0)         # 항등식 기여도
g = TOT.pct_change(12)
assert np.allclose(c.dropna().sum(axis=1), g.dropna())
def phase_of(ym):
    y = ym // 100
    for p, a, b in PHASES:
        if a <= y <= b: return p
    return None
ph = pd.Series([phase_of(x) for x in Wm.index], index=Wm.index)

rows = []
for p, a, b in PHASES:
    m = (ph == p) & g.notna()
    if m.sum() < 12: continue
    var = g[m].var()
    r = {"국면": PH_LABEL[p], "개월": int(m.sum()), "총수출 sd(%)": g[m].std()*100}
    for s in EXP15 + ["나머지"]:
        r[f"{s} 분산몫(%)"] = c.loc[m, s].cov(g[m]) / var * 100
    rows.append(r)
TAB_VAR = pd.DataFrame(rows).set_index("국면")
TAB_VAR.to_csv(os.path.join(OUT, "tab_variance_share.csv"), encoding="utf-8-sig")
print(TAB_VAR.round(1).T.to_string())

# 품목군 Δ12(계절조정) 표준편차와 쌍 상관 평균, 국면별
D12 = pd.DataFrame({s: SA[s]["sa_B"].diff(12) if s not in NO_SA else SA[s]["ln_c"].diff(12) for s in EXP15})
rows = []
for p, a, b in PHASES:
    m = (ph.reindex(D12.index) == p)
    d = D12[m].dropna()
    if len(d) < 12: continue
    C_ = d.corr().values; iu = np.triu_indices_from(C_, 1)
    r = {"국면": PH_LABEL[p], "쌍상관 평균": C_[iu].mean(), "반도체-반도체 제외 상관": d["반도체"].corr(SA["반도체 제외"]["sa_B"].diff(12)[m].dropna())}
    r.update({f"{s} sd(%)": d[s].std()*100 for s in EXP15})
    rows.append(r)
TAB_SYNC = pd.DataFrame(rows).set_index("국면")
TAB_SYNC.to_csv(os.path.join(OUT, "tab_sync.csv"), encoding="utf-8-sig")
print(TAB_SYNC.round(2).T.to_string())

국면              1995~2000  2001~2007  2008~2015  2016~2019  2020~2023  2024~2026.07
개월                   60.0       84.0       96.0       48.0       48.0          31.0
총수출 sd(%)            12.9       14.8       17.0       12.6       17.4          20.4
석유제품 분산몫(%)           7.2        5.7       15.8       10.7       14.4           7.0
화공품 분산몫(%)            7.5        7.4       11.6       11.6       15.6           4.4
의약품 분산몫(%)            0.1        0.1       -0.1        0.3        0.9           0.1
철강제품 분산몫(%)           3.8        7.5        8.9        9.3        9.9           1.8
일반기계 분산몫(%)           4.5        5.3        7.5        5.0        3.7           1.2
정밀기기 분산몫(%)           0.6        2.0        2.2        1.9        1.6           0.3
제조장비 분산몫(%)           0.2        0.6        0.7        2.4        0.6           0.2
가전제품 분산몫(%)           6.0        3.6        3.6       -0.7        2.1           0.4
컴퓨터주변기기 분산몫(%)       12.5        6.3        1.6        1.1        2.9       

In [15]:
# 그림 2: 반도체 대 반도체 제외 Δ12의 60개월 이동 상관
a_ = SA["반도체"]["sa_B"].diff(12); b_ = SA["반도체 제외"]["sa_B"].diff(12)
roll = a_.rolling(60).corr(b_).dropna()
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(tdate(roll.index), roll.values, color="black", lw=1.6)
ax.axhline(0, color="black", lw=0.8)
for ym in (200101, 200801, 201601, 202001, 202401):
    ax.axvline(tdate([ym])[0], color="0.5", lw=0.8, ls=":")
ax.set_ylabel("60개월 이동 상관"); ax.set_ylim(-1, 1)
style(ax); fig.tight_layout(); fig.savefig(os.path.join(IMG, "fig2_rolling_corr.png"), dpi=300); plt.close(fig)
ROLL = roll
print("이동 상관: 최소 %.2f @%d, 최대 %.2f @%d, 끝 %.2f @%d" % (roll.min(), roll.idxmin(), roll.max(), roll.idxmax(), roll.iloc[-1], roll.index[-1]))

이동 상관: 최소 -0.01 @201511, 최대 0.86 @200512, 끝 0.66 @202607


## §10. 본문 인용 수치의 검증

`무역_트렌드_기술적_분석.md`가 인용한 수치를 노트북 객체와 대조한다. `chk(이름, 계산값, 본문값, 자릿수)`는 반올림한 두 값이 다르면 모아 두었다가 마지막에 AssertionError로 멈춘다.
본문을 고치면 여기도 함께 고친다.

In [16]:
FAIL = []; N_CHK = [0]
def chk(name, calc, doc, nd=1):
    N_CHK[0] += 1
    c, d = round(float(calc), nd), round(float(doc), nd)
    if abs(c - d) > 10**(-nd) / 2 + 1e-9:
        FAIL.append(f"{name}: 계산 {c} ≠ 본문 {d}")

# --- 요약·I·II.1·II.2 ---
chk("총수출 2025 억달러", W.loc[202501:202512, "총수출"].sum()/1e8, 7093.0, 0)
chk("총수출 TTM 2026.07 억달러", W.loc[202508:202607, "총수출"].sum()/1e8, 9090.0, 0)
chk("총수출 2015 억달러", W.loc[201501:201512, "총수출"].sum()/1e8, 5268.0, 0)
chk("개월 수", len(W), 379, 0); chk("수출 코드 수", len(MAP), 153, 0); chk("나머지 코드 수", (MAP.group15 == "나머지").sum(), 96, 0)
tot3 = S[(S.yyyymm >= 202301) & (S.yyyymm <= 202512)]
tsum = tot3[tot3.series == "총수출"].dlr.sum()
chk("표1 총수출 2023~25 억달러", tsum/1e8, 20252.0, 0)
for s, v, p in [("반도체", 4197, 20.7), ("화공품", 2237, 11.0), ("승용차", 2051, 10.1), ("일반기계", 1533, 7.6), ("석유제품", 1491, 7.4),
                ("철강제품", 1440, 7.1), ("선박", 757, 3.7), ("자동차부품", 642, 3.2), ("무선통신기기", 538, 2.7), ("컴퓨터주변기기", 390, 1.9),
                ("정밀기기", 336, 1.7), ("이차전지", 310, 1.5), ("제조장비", 281, 1.4), ("의약품", 248, 1.2), ("가전제품", 234, 1.2), ("나머지", 3567, 17.6)]:
    v_ = tot3[tot3.series == s].dlr.sum()
    chk(f"표1 {s} 금액", v_/1e8, v, 0); chk(f"표1 {s} 비중", v_/tsum*100, p, 1)
chk("열다섯 커버리지", tot3[tot3.series.isin(EXP15)].dlr.sum()/tsum*100, 82.4, 1)
# II.2 본문(2026-09-14 추가): 코드 단위로 늘어놓을 때의 개수, 품목군 안에서 이름에 "기타"가 든 코드
for g_, n_ in [("반도체", 8), ("석유제품", 4), ("철강제품", 11)]: chk(f"II.2 {g_} 코드 수", (MAP.group15 == g_).sum(), n_, 0)
chk("II.2 품목군 안 기타 코드 수", (MAP.name_ko.str.contains("기타", na=False) & (MAP.group15 != "나머지")).sum(), 9, 0)

# --- II.3: 표2와 본문 ---
T2 = {"총수출": ("(112)(011)", 0.36, 10.5, -0.041, -0.046, 2, 0.35, 23.5, 14.3, 1.4), "반도체 제외": ("(110)(011)", 0.44, 12.5, -0.029, -0.051, 3, 0.38, 27.5, 12.5, 1.0),
      "반도체": ("(311)(011)", 0.10, 2.1, -0.020, -0.011, 1, 0.48, 17.9, 29.0, 4.5), "화공품": ("(311)(011)", 0.46, 12.9, -0.004, -0.023, 2, 0.50, 15.4, 12.7, 1.0),
      "승용차": ("(111)(011)", 0.52, 5.5, -0.050, -0.074, 8, 0.42, 51.3, 23.6, 3.2), "석유제품": ("(011)(011)", 0.54, 5.3, 0.002, -0.036, 2, 0.88, 22.8, 16.8, 3.8),
      "일반기계": ("(011)(011)", 0.49, 9.4, -0.010, -0.053, 4, 0.37, 41.7, 16.6, 1.7), "철강제품": ("(011)(011)", 0.59, 12.6, -0.009, 0.004, 5, 0.54, 16.2, 8.7, 1.6),
      "선박": ("(011)(011)*", -0.01, -0.0, -0.108, -0.153, 1, 1.08, 152.3, 62.5, 8.1), "자동차부품": ("(110)(011)", 0.68, 11.6, -0.017, -0.016, 12, 0.56, 31.2, 13.2, 5.2),
      "컴퓨터주변기기": ("(010)(011)", 0.24, 3.9, -0.031, -0.030, 5, 0.86, 31.3, 33.0, 4.9), "무선통신기기": ("(111)(101)", 0.34, 3.6, -0.017, -0.079, 3, 0.42, 35.6, 40.8, 3.1),
      "정밀기기": ("(010)(011)", 0.55, 11.1, -0.011, -0.059, 3, 0.41, 35.0, 21.6, 2.4), "이차전지": ("(011)(011)", 0.41, 8.4, -0.077, -0.024, 4, 0.51, 16.1, 19.2, 1.6),
      "제조장비": ("(111)(011)", 0.77, 4.2, 0.244, 0.049, 11, 0.96, 74.9, 39.8, 14.0), "의약품": ("(011)(011)", 0.06, 0.6, -0.016, -0.138, 5, 0.47, 29.2, 31.3, 3.7),
      "가전제품": ("(010)(011)", 0.55, 11.7, -0.043, -0.007, 7, 0.41, 23.9, 19.9, 2.0), "나머지": ("(011)(011)", 0.44, 13.7, -0.017, -0.034, 4, 0.28, 24.0, 15.1, 1.2)}
for s, (ar, b, t, sd_, cd_, no, m7, a0, a1, _rms) in T2.items():
    r = TAB_SA.loc[s]
    if r["ARIMA"] != ar: FAIL.append(f"표2 {s} ARIMA {r['ARIMA']} ≠ {ar}")
    chk(f"표2 {s} β", r["β_D"], b, 2); chk(f"표2 {s} t", r["t"], t, 1); chk(f"표2 {s} 설", r["설dur"], sd_, 3); chk(f"표2 {s} 추석", r["추석dur"], cd_, 3)
    chk(f"표2 {s} 이상치", r["이상치 수"], no, 0); chk(f"표2 {s} M7", r["M7"], m7, 2)
    chk(f"표2 {s} 진폭초기", r["계절진폭B 초기(%)"], a0, 1); chk(f"표2 {s} 진폭말기", r["계절진폭B 말기(%)"], a1, 1)
if sorted(NO_SA) != ["선박"]: FAIL.append(f"NO_SA {sorted(NO_SA)} ≠ ['선박']")
chk("A AICc", SPEC_A["aicc"], 16500.9, 1); chk("B AICc", SPEC_B["aicc"], 16490.9, 1); chk("ΔAICc", SPEC_A["aicc"] - SPEC_B["aicc"], 10.0, 1)
chk("B 화요일", SPEC_B["td"]["Tue"][0], 0.009, 3); chk("B 화요일 t", SPEC_B["td"]["Tue"][1], 2.3, 1)
chk("B n_hol", SPEC_B["user"]["n_hol"][0], -0.015, 3); chk("B n_hol t", SPEC_B["user"]["n_hol"][1], -5.5, 1)
chk("A 하루 환산", SPEC_A["coefs"]["ln_wd"][0] / 21, 0.017, 3); chk("A ln_wd", SPEC_A["coefs"]["ln_wd"][0], 0.364, 3)
if SPEC_A["td_peak"] or SPEC_B["td_peak"]: FAIL.append("요일 봉우리 경고가 있다")
chk("B 유의 요일 수", sum(abs(v[1]) >= 1.96 for v in SPEC_B["td"].values()), 1, 0)
chk("선박 D8 F", TAB_SA.loc["선박", "D8 F안정"], 6.1, 1); chk("D8 F 최소", TAB_SA["D8 F안정"].min(), 6.1, 1)
chk("QS조정 p<1 계열 수", (TAB_SA["QS조정 p"] < 0.999).sum(), 1, 0); chk("컴퓨터 QS조정 p", TAB_SA.loc["컴퓨터주변기기", "QS조정 p"], 0.30, 2)
if [o[0] for o in X13R["총수출"]["outliers"]] != ["LS2008.Nov", "LS2020.Apr"]: FAIL.append("총수출 이상치")
chk("이상치 있는 계열 수", (TAB_SA["이상치 수"] > 0).sum(), 23, 0)

# --- II.4: 표3 ---
for s, (per, cr, sh, su, ss, sr, nh, nu) in {"총수출": (44.7, 0.98, 4.4, 4.7, 10.7, 14.8, 2, 2), "반도체 제외": (45.0, 0.99, 4.9, 5.1, 9.3, 13.5, 2, 2),
                                            "반도체": (50.1, 0.58, 5.2, 0.8, 20.4, 31.4, 0, 0), "총수입": (42.4, 0.93, 5.0, 5.0, 14.3, 19.9, 2, 2)}.items():
    r = TAB_TR.loc[s]
    chk(f"표3 {s} 주기", r["UCM 순환주기(월)"], per, 1); chk(f"표3 {s} corr", r["corr(UCM,HP)"], cr, 2)
    chk(f"표3 {s} sdH", r["sd g_HP(%)"], sh, 1); chk(f"표3 {s} sdU", r["sd g_UCM(%)"], su, 1)
    chk(f"표3 {s} sdR", r["sd g_sa(%)"], sr, 1); chk(f"표3 {s} nH", r["부호전환 HP"], nh, 0); chk(f"표3 {s} nU", r["부호전환 UCM"], nu, 0)

# --- III.1 HP 트렌드 성장률 ---
def gext(s): return TR[s].g_main.dropna()*100
g = gext("총수출"); chk("총수출 g 정점", g.max(), 14.8); chk("총수출 g 정점 시점", g.idxmax(), 200501, 0)
chk("총수출 g 저점", g.min(), -0.5); chk("총수출 g 저점 시점", g.idxmin(), 201511, 0); chk("총수출 g 끝", g.iloc[-1], 10.2); chk("총수출 g 2020.01", g.loc[202001], 1.7)
g = gext("반도체 제외"); chk("제외 g 정점", g.max(), 14.9); chk("제외 g 정점 시점", g.idxmax(), 200501, 0)
chk("제외 g 저점", g.min(), -2.3); chk("제외 g 저점 시점", g.idxmin(), 201602, 0); chk("제외 g 끝", g.iloc[-1], 4.1)
g = gext("반도체"); chk("반도체 g 저점", g.min(), 1.3); chk("반도체 g 저점 시점", g.idxmin(), 200102, 0); chk("반도체 g 끝", g.iloc[-1], 26.4); chk("반도체 g 정점 시점", g.idxmax(), 202607, 0)

# --- 표4 ---
T4 = {"총수출": (2.2, 201201, 0, []), "반도체 제외": (5.8, 201201, 2, [200207, 200811]), "반도체": (2.3, 200207, 1, [202401]),
      "화공품": (6.7, 202112, 4, [200911, 201111, 202009, 202209]), "승용차": (4.5, 200709, 3, [200712, 200912, 201204]),
      "석유제품": (3.4, 201203, 0, []), "일반기계": (9.1, 201209, 4, [200207, 200801, 201001, 201201]), "철강제품": (4.2, 201205, 2, [200210, 200811]),
      "선박": (7.0, 200907, 0, []), "자동차부품": (15.1, 201408, 5, [199801, 200207, 200711, 200911, 201111]),
      "컴퓨터주변기기": (1.5, 200011, 2, [202204, 202404]), "무선통신기기": (26.3, 200412, 1, [200412]), "정밀기기": (8.8, 202112, 2, [200309, 200602]),
      "이차전지": (7.5, 202109, 2, [199910, 201201]), "제조장비": (7.2, 200101, 1, [199807]), "의약품": (9.0, 200205, 3, [202001, 202206, 202407]),
      "가전제품": (1.9, 201408, 3, [200601, 200801, 201012]), "나머지": (5.3, 200207, 3, [200210, 201306, 202009]), "총수입": (1.5, 201203, 0, [])}
for s, (f, at, m, br) in T4.items():
    chk(f"표4 {s} supF", BP[s]["supF"], f, 1); chk(f"표4 {s} at", BP[s]["supF_at"], at, 0); chk(f"표4 {s} m", BP[s]["m_lwz"], m, 0)
    if BP[s]["br_lwz"] != br: FAIL.append(f"표4 {s} 단절 {BP[s]['br_lwz']} ≠ {br}")
chk("BIC 상한 5 계열 수", (TAB_BP["m(BIC)"] == 5).sum(), 13, 0)
# III.2 본문(2026-09-14 추가): 5% 임계값 8.85를 넘는 수출 계열, 반도체 제외·철강제품의 단절 시점
_over = sorted(s for s in T4 if s != "총수입" and BP[s]["supF"] > 8.85)
if _over != sorted(["일반기계", "자동차부품", "무선통신기기", "의약품"]): FAIL.append(f"III.2 8.85 초과 계열 {_over}")
chk("III.2 8.85 초과 계열 수", len(_over), 4, 0)
chk("III.2 반도체 제외 상향", BP["반도체 제외"]["br_lwz"][0], 200207, 0); chk("III.2 철강제품 상향", BP["철강제품"]["br_lwz"][0], 200210, 0)
chk("III.2 반도체 제외 하향", BP["반도체 제외"]["br_lwz"][1], 200811, 0); chk("III.2 철강제품 하향", BP["철강제품"]["br_lwz"][1], 200811, 0)

# --- 표5 국면 ---
T5 = {"반도체 제외": [(199601, 200206, 78, 3.5, 5.3, 11.7), (200207, 200810, 76, 17.1, 13.1, 7.2), (200811, 202607, 213, 2.2, 2.9, 13.6)],
      "반도체": [(199601, 202312, 336, 6.2, 7.5, 29.6), (202401, 202607, 31, 42.8, 21.0, 31.3)],
      "철강제품": [(199601, 200209, 81, -0.0, 3.6, 16.5), (200210, 200810, 73, 22.2, 16.5, 10.3), (200811, 202607, 213, 1.3, 2.3, 17.8)],
      "일반기계": [(200207, 200712, 66, 21.4, 17.6, 8.5), (201201, 202607, 175, 0.6, 1.6, 9.3)],
      "무선통신기기": [(199601, 200411, 107, 35.4, 36.3, 24.4), (200412, 202607, 260, -0.9, -0.7, 27.4)]}
for s, segs in T5.items():
    P_ = PH[PH.계열 == s].set_index("시작")
    for a, b, n, m1, m2, sd in segs:
        r = P_.loc[a]
        chk(f"표5 {s} {a} 끝", r["끝"], b, 0); chk(f"표5 {s} {a} 개월", r["개월"], n, 0)
        chk(f"표5 {s} {a} Δ12", r["Δ12 평균(%)"], m1, 1); chk(f"표5 {s} {a} HP", r["HP 트렌드 성장 평균(%)"], m2, 1); chk(f"표5 {s} {a} sd", r["Δ12 표준편차(%)"], sd, 1)

# --- 표6·7 ---
T6 = {1995: (2.4, 3.7, 24.3, 69.6, 14.1, 343, 23.1), 2000: (1.6, 6.7, 17.6, 74.1, 15.1, 406, 27.4), 2007: (1.0, 7.9, 7.4, 83.7, 10.5, 396, 24.4),
      2015: (1.3, 7.5, 6.7, 84.5, 12.0, 368, 22.8), 2019: (1.5, 9.0, 6.3, 83.2, 17.8, 420, 27.2), 2023: (1.7, 9.9, 5.3, 83.1, 15.9, 422, 26.6),
      2024: (1.7, 8.9, 5.0, 84.4, 21.0, 478, 29.8), 2025: (1.7, 7.9, 5.0, 85.4, 24.7, 563, 32.6), 2026: (1.4, 7.4, 4.5, 86.7, 35.3, 1027, 41.0)}
for y, (f, r_, l, h, se, hh_, t3) in T6.items():
    row = TAB_SHARE.loc[y]
    chk(f"표6 {y} 식료", row["식료 및 직접소비재"], f); chk(f"표6 {y} 원료", row["원료 및 연료"], r_); chk(f"표6 {y} 경공업", row["경공업품"], l)
    chk(f"표6 {y} 중화학", row["중화학 공업품"], h); chk(f"표6 {y} 반도체", row["반도체(%)"], se); chk(f"표6 {y} HHI", row["HHI"], hh_, 0); chk(f"표6 {y} top3", row["top3(%)"], t3)
chk("HHI 1995~2016 최소", TAB_SHARE.loc[1995:2016, "HHI"].min(), 329, 0); chk("HHI 1995~2016 최대", TAB_SHARE.loc[1995:2016, "HHI"].max(), 406, 0)
chk("HHI 2018", TAB_SHARE.loc[2018, "HHI"], 491, 0); chk("반도체 2018", TAB_SHARE.loc[2018, "반도체(%)"], 21.4); chk("HHI 2017", TAB_SHARE.loc[2017, "HHI"], 425, 0)
chk("HHI 2019~2023 최소", TAB_SHARE.loc[2019:2023, "HHI"].min(), 410, 0); chk("HHI 2019~2023 최대", TAB_SHARE.loc[2019:2023, "HHI"].max(), 427, 0)
chk("유효 코드 수 1995", 10000 / TAB_SHARE.loc[1995, "HHI"], 29, 0); chk("유효 코드 수 2018", 10000 / TAB_SHARE.loc[2018, "HHI"], 20, 0)
chk("유효 코드 수 2026", 10000 / TAB_SHARE.loc[2026, "HHI"], 10, 0)
chk("유효 코드 수 1995~2016 최소", 10000 / TAB_SHARE.loc[1995:2016, "HHI"].max(), 25, 0); chk("유효 코드 수 1995~2016 최대", 10000 / TAB_SHARE.loc[1995:2016, "HHI"].min(), 30, 0)
chk("균등 분포 HHI", 10000 / 153, 65, 0)
T7 = {"반도체": (14.1, 15.1, 10.5, 12.0, 17.8, 15.9, 35.3), "화공품": (7.0, 7.7, 9.7, 10.2, 11.6, 12.6, 8.0), "승용차": (5.2, 6.4, 9.3, 7.9, 7.5, 10.8, 7.5),
      "석유제품": (1.9, 5.4, 6.5, 6.1, 7.6, 8.3, 6.1), "일반기계": (5.6, 5.0, 7.3, 8.3, 8.8, 8.4, 5.3), "철강제품": (8.0, 6.6, 8.5, 7.9, 8.1, 7.9, 5.3),
      "선박": (4.4, 4.8, 7.2, 7.4, 3.6, 3.3, 3.7), "자동차부품": (0.7, 1.2, 3.3, 4.9, 4.0, 3.5, 2.1), "컴퓨터주변기기": (3.6, 6.9, 3.6, 1.3, 1.6, 1.4, 3.8),
      "무선통신기기": (0.8, 4.1, 7.9, 2.8, 4.0, 3.0, 2.2), "정밀기기": (0.6, 0.9, 1.9, 1.6, 2.1, 1.8, 1.2), "이차전지": (0.5, 0.6, 0.7, 1.1, 1.7, 1.9, 1.1),
      "제조장비": (0.0, 0.2, 0.5, 1.0, 1.6, 1.3, 1.3), "의약품": (0.2, 0.2, 0.2, 0.4, 0.8, 1.1, 1.2), "가전제품": (6.5, 5.1, 1.5, 2.1, 1.6, 1.3, 0.8),
      "나머지": (40.8, 29.8, 21.2, 25.0, 17.7, 17.6, 15.3)}
for s, vals in T7.items():
    for y, v in zip([1995, 2000, 2007, 2015, 2019, 2023, 2026], vals): chk(f"표7 {s} {y}", sh_g.loc[y, s], v)

# --- 표8 기여도 ---
T8 = {"총수출 증가율(%)": (37.8, 115.6, 41.8, 2.9, 16.6, 43.8), "연평균(%)": (6.6, 11.6, 4.5, 0.7, 3.9, 15.1), "반도체": (1.34, 1.08, 0.82, 1.58, 0.19, 13.47), "화공품": (0.72, 1.89, 0.59, 0.45, 0.77, -0.45), "승용차": (0.73, 1.94, 0.24, -0.06, 1.28, -0.01), "석유제품": (1.10, 1.23, 0.27, 0.42, 0.51, 0.19), "일반기계": (0.26, 1.55, 0.55, 0.19, 0.26, -0.34), "철강제품": (0.22, 1.68, 0.33, 0.13, 0.26, -0.12), "선박": (0.43, 1.54, 0.40, -0.91, 0.06, 0.81), "자동차부품": (0.19, 0.86, 0.44, -0.18, 0.03, -0.17), "컴퓨터주변기기": (1.17, 0.14, -0.22, 0.07, 0.01, 1.60), "무선통신기기": (0.96, 1.84, -0.48, 0.32, -0.12, 0.04), "정밀기기": (0.13, 0.45, 0.04, 0.14, 0.01, -0.02), "이차전지": (0.06, 0.15, 0.10, 0.15, 0.13, -0.11), "제조장비": (0.07, 0.12, 0.12, 0.16, -0.04, 0.21), "의약품": (0.01, 0.04, 0.05, 0.09, 0.11, 0.23), "가전제품": (0.11, -0.26, 0.18, -0.10, -0.03, -0.07), "나머지": (0.06, 2.28, 1.78, -1.71, 0.72, 1.69)}
labels = list(TAB_CONTRIB.index)
for k, vals in T8.items():
    nd = 1 if k.endswith("(%)") else 2
    for lab, v in zip(labels, vals): chk(f"표8 {k} {lab}", TAB_CONTRIB.loc[lab, k], v, nd)
chk("2024~ 반도체 몫", TAB_CONTRIB.loc[labels[-1], "반도체"] / TAB_CONTRIB.loc[labels[-1], EXP15 + ["나머지"]].sum() * 100, 79.5, 1)
chk("2024~ 기여도 합", TAB_CONTRIB.loc[labels[-1], EXP15 + ["나머지"]].sum(), 16.95, 2)
chk("햇수 마지막", TAB_CONTRIB.loc[labels[-1], "햇수"], 2.58, 2)

# --- 표9 단가·물량 ---
PQ = TAB_PQ.set_index(["계열", "국면"])
T9 = {"총수출": [(-2.5, 8.9), (5.7, 5.3), (3.3, 1.1), (1.0, -0.2), (4.3, -0.4), (13.5, 0.6)], "반도체 제외": [(-1.9, 8.0), (6.0, 5.7), (2.3, 1.9), (-0.3, -0.7), (5.6, -1.2), (3.6, 0.3)], "반도체": [(-6.3, 14.1), (3.2, 2.6), (10.9, -4.9), (8.2, 2.4), (-2.4, 3.4), (39.8, 5.0)], "화공품": [(-6.1, 14.4), (7.6, 6.8), (0.3, 4.6), (0.9, 3.2), (6.2, -0.3), (-4.6, 0.8)], "승용차": [(-3.5, 14.1), (4.2, 12.0), (2.0, 0.3), (1.1, -1.9), (6.6, 6.5), (-2.5, 2.4)], "석유제품": [(8.4, 18.5), (12.8, 0.8), (-3.0, 6.6), (4.0, 2.1), (7.9, -2.0), (2.1, 0.2)], "일반기계": [(-6.5, 10.7), (4.1, 12.3), (3.4, 2.4), (1.5, 0.7), (3.0, -0.3), (0.4, -4.6)], "철강제품": [(-4.3, 6.9), (9.2, 5.4), (-2.4, 5.7), (2.2, -0.6), (5.2, -2.2), (-1.3, -0.3)], "선박": [(24.4, -16.5), (13.7, 3.2), (1.8, 2.8), (-8.9, -8.2), (-1.9, 3.5), (13.5, 5.5)], "자동차부품": [(-3.4, 19.8), (5.1, 20.3), (1.9, 7.1), (-1.7, -2.5), (2.7, -2.0), (-3.5, -1.7)], "컴퓨터주변기기": [(7.9, 11.3), (8.0, -6.2), (7.3, -15.5), (3.3, 1.6), (5.3, -4.8), (55.5, -1.6)], "무선통신기기": [(16.1, 22.5), (9.1, 11.2), (1.2, -9.6), (15.2, -5.9), (14.3, -17.5), (-4.8, 6.1)], "정밀기기": [(0.6, 13.6), (8.1, 13.3), (-7.5, 9.5), (6.0, 1.5), (10.4, -9.8), (4.6, -5.7)], "이차전지": [(2.9, 7.3), (6.5, 8.8), (0.1, 9.0), (5.9, 5.3), (4.2, 2.5), (-10.7, 4.0)], "제조장비": [(-56.0, 158.9), (5.3, 16.0), (-3.1, 16.6), (1.1, 11.0), (4.4, -6.8), (12.8, 1.2)], "의약품": [(2.0, 3.2), (6.9, 5.7), (9.2, 3.9), (10.3, 4.9), (8.4, 3.1), (15.2, 2.0)], "가전제품": [(-4.8, 6.4), (6.3, -12.7), (5.4, 2.9), (2.6, -8.1), (3.8, -5.7), (-4.4, -1.8)], "나머지": [(-6.1, 6.2), (0.9, 5.2), (6.4, 0.0), (-5.6, -2.4), (5.3, -1.5), (7.6, 1.0)]}
for s, vals in T9.items():
    for lab, (p_, q_) in zip(labels, vals):
        chk(f"표9 {s} {lab} P", PQ.loc[(s, lab), "ΔlnP 연평균(%)"], p_); chk(f"표9 {s} {lab} Q", PQ.loc[(s, lab), "ΔlnQ 연평균(%)"], q_)
# V 본문(2026-09-14 추가): 2016년 이후 국면 3개의 총수출 물량
for lab, v in zip(labels[-3:], (-0.2, -0.4, 0.6)): chk(f"V 총수출 물량 {lab}", PQ.loc[("총수출", lab), "ΔlnQ 연평균(%)"], v)

# --- 표10 ---
T10 = {"총수출 sd(%)": (12.9, 14.8, 17.0, 12.6, 17.4, 20.4), "반도체 분산몫(%)": (20.6, 22.7, 9.3, 35.6, 22.7, 65.9), "나머지 분산몫(%)": (24.1, 19.0, 13.9, 16.7, 12.4, 5.6),
       "석유제품 분산몫(%)": (7.2, 5.7, 15.8, 10.7, 14.4, 7.0), "화공품 분산몫(%)": (7.5, 7.4, 11.6, 11.6, 15.6, 4.4), "철강제품 분산몫(%)": (3.8, 7.5, 8.9, 9.3, 9.9, 1.8),
       "컴퓨터주변기기 분산몫(%)": (12.5, 6.3, 1.6, 1.1, 2.9, 9.3)}
for k, vals in T10.items():
    for lab, v in zip(labels, vals): chk(f"표10 {k} {lab}", TAB_VAR.loc[lab, k], v)
for k, vals, nd in [("반도체-반도체 제외 상관", (0.52, 0.83, 0.53, 0.78, 0.65, 0.91), 2), ("쌍상관 평균", (0.22, 0.30, 0.40, 0.11, 0.32, 0.20), 2)]:
    for lab, v in zip(labels, vals): chk(f"표10 {k} {lab}", TAB_SYNC.loc[lab, k], v, nd)
chk("이동상관 최대", ROLL.max(), 0.86, 2); chk("이동상관 최대 시점", ROLL.idxmax(), 200512, 0)
chk("이동상관 최소", ROLL.min(), -0.01, 2); chk("이동상관 최소 시점", ROLL.idxmin(), 201511, 0); chk("이동상관 끝", ROLL.iloc[-1], 0.66, 2)
chk("이동상관 첫 시점", ROLL.index[0], 200012, 0)
# VI 본문(2026-09-14 추가): 표 10의 2016~2019년 상관은 그 국면 48개월로 잰 값, 반도체 제외·철강제품 단절의 방향(표 5)
chk("VI 2016~2019 국면 개월", D12.loc[201601:201912, "반도체"].notna().sum(), 48, 0)
for s, (up, dn) in [("반도체 제외", (200207, 200811)), ("철강제품", (200210, 200811))]:
    _m = PH[PH.계열 == s].set_index("시작")["Δ12 평균(%)"]
    if not (_m.loc[up] > _m.iloc[0] and _m.loc[dn] < _m.loc[up]): FAIL.append(f"III.2 {s} 단절 방향(상향·하향)")

# --- II.2 본문: 나머지의 구성 (옛 부록 A) ---
_rest = MAP[MAP.group15 == "나머지"]
_x1 = _rest.x1.str.replace(r"^\d\.\s*", "", regex=True).value_counts()
chk("나머지 식료", _x1.get("식료 및 직접소비재", 0), 13, 0); chk("나머지 원료연료", _x1.get("원료 및 연료", 0), 7, 0)
chk("나머지 경공업", _x1.get("경공업품", 0), 61, 0); chk("나머지 중화학", _x1.get("중화학 공업품", 0), 15, 0)
_r3 = tot3[tot3.series == "나머지"].dlr.sum()
_five = con.execute("SELECT SUM(dlr) FROM fact_temper WHERE imexp='수출' AND yyyymm BETWEEN 202301 AND 202512 AND temper_cd IN ('44Z01','44203','44401','44404','4Z001')").fetchone()[0]
chk("나머지 다섯 코드 몫", _five / _r3 * 100, 44.8, 1)

print(f"검증 {N_CHK[0]}건, 실패 {len(FAIL)}건")
for f in FAIL: print("  ", f)
assert not FAIL, f"본문 수치 불일치 {len(FAIL)}건"
print("본문 인용 수치 검증 통과")

검증 953건, 실패 0건
본문 인용 수치 검증 통과
